In [27]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [28]:
load_dotenv()

True

In [29]:
model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash-lite')

In [30]:
class Evaluationschema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge = 0, le = 10)

In [31]:
structured_model = model.with_structured_output(Evaluationschema)

In [32]:
essay = """Artificial Intelligence (AI) is one of the most important technological developments of the modern world. It refers to the ability of machines and computer systems to perform tasks that usually require human intelligence, such as learning, problem-solving, decision-making, and understanding language. In recent years, AI has become an essential part of our daily lives and is transforming the way people work, study, communicate, and solve problems.

One of the major areas where artificial intelligence has made a significant impact is education. AI-powered applications help students understand difficult concepts, practice questions, and receive personalized learning recommendations. Students can use AI tools to improve their writing, learn programming, and prepare for examinations. Teachers can also use these technologies to reduce repetitive tasks and focus more on helping students learn effectively.

Another important application of AI is healthcare. Artificial intelligence can assist doctors in analyzing medical images, identifying potential diseases, and examining large amounts of patient data. It can help researchers discover new medicines and support hospitals in managing their resources. Although AI can improve healthcare services, it cannot completely replace the experience, judgment, and empathy of medical professionals.

Artificial intelligence also plays a significant role in business and industry. Companies use AI to automate repetitive tasks, analyze customer behavior, improve product recommendations, and provide customer support through chatbots. In manufacturing, AI-powered robots can perform tasks with speed and precision. These applications can increase productivity, reduce operational costs, and help organizations make better decisions.

AI has also changed our everyday lives. Virtual assistants, navigation applications, social media recommendations, online shopping platforms, and streaming services use AI technologies to provide personalized experiences. For example, navigation applications can analyze traffic conditions and suggest faster routes. Similarly, online shopping platforms recommend products based on users' interests and previous activities.

Despite its advantages, artificial intelligence has several disadvantages. Automation may reduce the demand for certain jobs, creating challenges for workers who need to develop new skills. AI systems can also produce incorrect information, make biased decisions, and create privacy concerns when personal data is not handled properly. Furthermore, excessive dependence on AI may reduce opportunities for independent thinking and problem-solving.

To address these challenges, AI should be developed and used responsibly. Governments, educational institutions, and businesses should establish appropriate guidelines to protect privacy, reduce bias, and ensure transparency. People should also learn new skills and understand the limitations of AI rather than blindly trusting every result it produces.

In conclusion, artificial intelligence has the potential to improve human life in many ways, from education and healthcare to business and communication. However, its benefits must be balanced against its risks. AI should be considered a tool that supports human intelligence rather than a complete replacement for human judgment and creativity. With responsible development and thoughtful use, artificial intelligence can contribute to a more productive, innovative, and efficient future."""

In [33]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10. Essay: {essay}'

print(structured_model.invoke(prompt))

feedback='The essay demonstrates an exceptional level of language quality, with clear, natural phrasing, accurate grammar, and sophisticated vocabulary. The structure is well-organized, with smooth transitions between paragraphs. The tone is formal, objective, and appropriate for an academic or professional discussion. Sentence variety is strong, contributing to a fluid and engaging reading experience.' score=10


In [34]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [35]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10. Essay: {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_score': [output.score]}

In [36]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10. Essay: {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_score': [output.score]}

In [37]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the clarity of thoughts of the following essay and provide a feedback and assign a score out of 10. Essay: {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_score': [output.score]}

In [38]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback {state["language_feedback"]} \n analysis feedback {state["analysis_feedback"]} \n clarity feedback {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content[0]['text']

    avg_score = sum(state['individual_score'])/len(state['individual_score'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [39]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [40]:
initial_state = {
    'essay': essay,
}

final_state = workflow.invoke(initial_state)

print(final_state)
# print(final_state['avg_score'])
# print(final_state['overall_feedback'])

{'essay': "Artificial Intelligence (AI) is one of the most important technological developments of the modern world. It refers to the ability of machines and computer systems to perform tasks that usually require human intelligence, such as learning, problem-solving, decision-making, and understanding language. In recent years, AI has become an essential part of our daily lives and is transforming the way people work, study, communicate, and solve problems.\n\nOne of the major areas where artificial intelligence has made a significant impact is education. AI-powered applications help students understand difficult concepts, practice questions, and receive personalized learning recommendations. Students can use AI tools to improve their writing, learn programming, and prepare for examinations. Teachers can also use these technologies to reduce repetitive tasks and focus more on helping students learn effectively.\n\nAnother important application of AI is healthcare. Artificial intelligen